In [4]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Dense, Activation, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import categorical_crossentropy
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model
from tensorflow.keras.applications import imagenet_utils
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix
import itertools
import os
import shutil
from sklearn import metrics
import random
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
train_path = "C:\\Users\\tishy\\OneDrive\\Desktop\\Smi\\Dataset\\Train"
valid_path = "C:\\Users\\tishy\\OneDrive\\Desktop\\Smi\\Dataset\\Validation"
test_path = "C:\\Users\\tishy\\OneDrive\\Desktop\\Smi\\Dataset\\Test"

train_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.xception.preprocess_input).flow_from_directory(
    directory=train_path, classes = ['BabyCry', 'Bark', 'Cough', 'Dishes', 'Doorbell', 'FireAlarm', 'SmokeDetector', 'Sneeze', 'Thunder', 'Water'], target_size=(224,224), batch_size=32)
valid_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.xception.preprocess_input).flow_from_directory(
    directory=valid_path, classes = ['BabyCry', 'Bark', 'Cough', 'Dishes', 'Doorbell', 'FireAlarm', 'SmokeDetector', 'Sneeze', 'Thunder', 'Water'], target_size=(224,224), batch_size=32)
test_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.xception.preprocess_input).flow_from_directory(
    directory=test_path, classes = ['BabyCry', 'Bark', 'Cough', 'Dishes', 'Doorbell', 'FireAlarm', 'SmokeDetector', 'Sneeze', 'Thunder', 'Water'], target_size=(224,224), batch_size=32, shuffle = False)

Found 4890 images belonging to 10 classes.
Found 524 images belonging to 10 classes.
Found 524 images belonging to 10 classes.


In [3]:
xception = tf.keras.applications.xception.Xception(include_top = False)
xception.summary()

Model: "xception"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, None, None,  0                                            
__________________________________________________________________________________________________
block1_conv1 (Conv2D)           (None, None, None, 3 864         input_1[0][0]                    
__________________________________________________________________________________________________
block1_conv1_bn (BatchNormaliza (None, None, None, 3 128         block1_conv1[0][0]               
__________________________________________________________________________________________________
block1_conv1_act (Activation)   (None, None, None, 3 0           block1_conv1_bn[0][0]            
___________________________________________________________________________________________

In [4]:
#add a global spatial average pooling layer
x = xception.output
x = GlobalAveragePooling2D()(x)
x = Dense(200,activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(170,activation='relu')(x)
predictions = Dense(10,activation='softmax')(x)
model = Model(inputs=xception.input, outputs=predictions)
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy',metrics=['accuracy'])


In [5]:
model.fit(x=train_batches,
            steps_per_epoch=len(train_batches),
            validation_data=valid_batches,
            validation_steps=len(valid_batches),
            epochs=15,
) 

Epoch 1/15
153/153 [==============================] - 1548s 10s/step - loss: 1.6871 - accuracy: 0.4103 - val_loss: 318.7954 - val_accuracy: 0.3435
Epoch 2/15
153/153 [==============================] - 1476s 10s/step - loss: 0.1688 - accuracy: 0.9562 - val_loss: 10.8745 - val_accuracy: 0.4561
Epoch 3/15
153/153 [==============================] - 1478s 10s/step - loss: 0.0764 - accuracy: 0.9820 - val_loss: 0.6185 - val_accuracy: 0.8588
Epoch 4/15
153/153 [==============================] - 1476s 10s/step - loss: 0.0407 - accuracy: 0.9893 - val_loss: 1.1163 - val_accuracy: 0.8244
Epoch 5/15
153/153 [==============================] - 1474s 10s/step - loss: 0.0948 - accuracy: 0.9801 - val_loss: 0.4017 - val_accuracy: 0.9256
Epoch 6/15
153/153 [==============================] - 1477s 10s/step - loss: 0.0767 - accuracy: 0.9841 - val_loss: 0.1189 - val_accuracy: 0.9618
Epoch 7/15
153/153 [==============================] - 1472s 10s/step - loss: 0.0197 - accuracy: 0.9942 - val_loss: 2.1868e-04 -

In [6]:
predictions = model.predict(x=test_batches, verbose=1)

17/17 [==============================] - 23s 1s/step


In [7]:
from sklearn import metrics
metrics.accuracy_score(test_batches.classes, np.argmax(predictions, axis = -1))

1.0

In [2]:
from tensorflow.keras.models import load_model
model2 = load_model("D:\\SavedModels\\XCLatest.h5")